# MediCheck-GPT: run the full flow on Colab (v3)
Runtime → Change runtime type → **T4 GPU**. Run top to bottom.

**Before you start:** Runtime → *Disconnect and delete runtime*, so no old copy of the code is left behind. **When you save:** File → *Save a copy in Drive*. Do not save back to GitHub over this file; send the outputs to Claude instead.

## 0. Setup: GPU, code, data

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # confirm the T4 is attached

name, memory.total [MiB]
Tesla T4, 15360 MiB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')                     # the raw CMS file lives in my Drive
ROOT = '/content/drive/MyDrive/build-transformer'   # same folder as 01_data.ipynb

In [ ]:
!rm -rf /content/medicheck-gpt                       # never run an old copy of the code
!git clone https://github.com/cece15/medicheck-gpt.git /content/medicheck-gpt
%cd /content/medicheck-gpt
!pip install -q -r requirements.txt
# data and checkpoints live in Drive, so a Colab disconnect loses nothing: rerun from the cell that stopped
!mkdir -p "$ROOT/medicheck/data" "$ROOT/medicheck/checkpoints"
!ln -sfn "$ROOT/medicheck/data" data && ln -sfn "$ROOT/medicheck/checkpoints" checkpoints && ls -l
!git log --oneline -1                               # should NOT say 'Created using Colab'

## 1. Data: CSV → one sentence per row → split by hospital

In [ ]:
import os
if not os.path.exists('data/processed/test.txt'):
    !python gpt.py --input "$ROOT/data/raw/MUP_INP_RY26_P03_V10_DY24_PrvSvc.CSV" --prepare data/processed
!head -c 600 data/processed/train.txt

## 2. Baseline: bigram (one character of memory)

In [ ]:
if not os.path.exists('checkpoints/bigram.pth'):
    !python gpt.py --input data/processed --train checkpoints/bigram.pth --model bigram --context_size 8 --lr 1e-2 --steps 3000

## 3. Transformer: 4 layers, 4 heads, 128-dim, context 271 (one full record)
Cosine LR with warmup, weight decay, early stopping. First run (3 Nov) overfit after ~1,500 steps, so these guards matter.

In [ ]:
if not os.path.exists('checkpoints/gpt.pth'):
    !python gpt.py --input data/processed --train checkpoints/gpt.pth --steps 10000 --eval_every 500 --patience 4

In [ ]:
# Loss curve for the slides
import json, matplotlib.pyplot as plt
h = json.load(open('checkpoints/gpt_history.json'))
plt.plot([r['step'] for r in h], [r['train'] for r in h], label='train')
plt.plot([r['step'] for r in h], [r['val'] for r in h], label='validation (unseen hospitals)')
plt.xlabel('step'); plt.ylabel('cross-entropy loss'); plt.legend(); plt.title('MediCheck-GPT training')
plt.savefig('checkpoints/loss_curve.png', dpi=200, bbox_inches='tight'); plt.show()

## 3a. Hyperparameter comparison (Week 7: fine-tuning and hyperparameter tuning)
Same data, same seed, same early stopping. Only the model size and dropout change. About 15 minutes.

In [ ]:
# smaller and larger than the base model (base = 4 layers, 4 heads, 128-dim, already trained above)
if not os.path.exists('checkpoints/gpt_large.pth'):
    !python gpt.py --input data/processed --train checkpoints/gpt_small.pth --n_layer 2 --n_head 2 --n_embd 64 --steps 6000
    !python gpt.py --input data/processed --train checkpoints/gpt_large.pth --n_layer 6 --n_head 6 --n_embd 192 --dropout 0.2 --steps 6000

In [ ]:
# one table for the slides: which size generalizes best to unseen hospitals?
import math, torch, pandas as pd
from src.model import build, count_params
rows = []
for name in ['gpt_small', 'gpt', 'gpt_large']:
    ck = torch.load(f'checkpoints/{name}.pth', map_location='cpu')
    best = min(ck['history'], key=lambda r: r['val'])            # the checkpoint holds the best evaluation
    rows.append({'model': name, **{k: ck['config'][k] for k in ['n_layer', 'n_head', 'n_embd', 'dropout']},
                 'parameters': count_params(build(ck['kind'], **ck['config'])),
                 'best_step': best['step'], 'best_val_loss': round(best['val'], 4),
                 'val_perplexity': round(math.exp(best['val']), 3), 'seconds': best['seconds']})
pd.DataFrame(rows)

## 3b. Kacper's recommendation: an LLM teacher writes plain-English descriptions (v2)
Run 1 showed the teacher invents numbers (14%) and, worse, attaches real numbers to the wrong label. v2 gives the LLM each fact as a full phrase and one example, rejects sentences with invented or mislabeled numbers, and gives each rejected record one retry. About 25–30 minutes.

In [ ]:
import os
if not os.path.exists('data/described_v2/test.txt'):       # the teacher is the slow step: never run it twice
    !python gpt.py --input data/processed --teacher data/described_v2 --limit 5000
print(open('data/described_v2/teacher_report.json').read())
!head -c 1200 data/described_v2/train.txt

## 3c. Student v2: pre-train, then fine-tune (Lecture 6–7)
Run 1 trained from scratch on random windows; it learned the teacher's style but ignored the record. v2 starts from the record model (`gpt.pth`, which already knows records) and fine-tunes on whole examples with the loss on the explanation only. Validation loss here is explanation-only, so it is not comparable with run 1's.

In [ ]:
if not os.path.exists('checkpoints/explain_v2.pth'):
    !python gpt.py --input data/described_v2 --train checkpoints/explain_v2.pth --init_from checkpoints/gpt.pth --steps 4000 --eval_every 250 --patience 4

In [ ]:
# pass the record as a Python argument: records contain '$', which the shell would eat
import subprocess
rec = open('data/processed/test.txt').read().split('\n')[1]
print('record :', rec)
print(subprocess.run(['python', 'gpt.py', '--eval', 'checkpoints/explain_v2.pth', '--prompt', rec, '--n', '3', '--temperature', '0.5'],
                     capture_output=True, text=True).stdout)

## 3d. Student v3: placeholders (run 3)
Run 2's student read the record's style but wrote numbers from memory: 0% copied. v3 trains on the teacher's sentences with the record's values replaced by placeholders (`{BILL}`, `{TOTAL}`, `{HOSPITAL}`...); the student learns where each one goes and Python puts the real values back. About 15 minutes.

In [ ]:
if not os.path.exists('checkpoints/explain_v3.pth'):
    !python gpt.py --input data/described_v2 --train checkpoints/explain_v3.pth --init_from checkpoints/gpt.pth --delex --steps 4000 --eval_every 250 --patience 4
print(subprocess.run(['python', 'gpt.py', '--eval', 'checkpoints/explain_v3.pth', '--prompt', rec, '--n', '3', '--temperature', '0.5'],
                     capture_output=True, text=True).stdout)

## 4. Generate

In [ ]:
!python gpt.py --eval checkpoints/gpt.pth --n 3
!python gpt.py --eval checkpoints/gpt.pth --n 2 --prompt "Trinity Health Livonia Hospital in Livonia, MI billed DRG "

## 5. Evaluate: perplexity, token accuracy, ROUGE-L (+ BLEU, BERTScore), record validity

In [ ]:
# record models (bigram and GPT were evaluated in run 2; skip if the reports exist)
if not os.path.exists('checkpoints/gpt_eval.json'):
    !python gpt.py --input data/processed --evaluate checkpoints/bigram.pth
    !python gpt.py --input data/processed --evaluate checkpoints/gpt.pth --hf_metrics
# explainers: v2 (copy numbers itself) vs v3 (placeholders), same test sentences, same checks
!python gpt.py --input data/described_v2 --evaluate checkpoints/explain_v2.pth --hf_metrics
!python gpt.py --input data/described_v2 --evaluate checkpoints/explain_v3.pth --hf_metrics

## 6. Flag silent errors
AUC across records (raw surprise; surprise after the name; ratio to the bigram) and the paired test: on the same record, does the planted error raise surprise? 0.5 = chance, below 0.5 = backwards. Runs on CPU too.

In [ ]:
!python gpt.py --input data/processed --flag checkpoints/gpt.pth --background checkpoints/bigram.pth
!python gpt.py --input data/processed --flag checkpoints/gpt_large.pth --background checkpoints/bigram.pth

## 6b. Demo file for Tuesday: five real test records and five with planted errors, shuffled. Rules catch arithmetic; the model ranks the rest.

In [ ]:
import random, sys
sys.path.insert(0, '.')
from src.flag import corrupt, KINDS
from src.data import parse
test = [r for r in open('data/processed/test.txt').read().split('\n') if parse(r)]
train = [r for r in open('data/processed/train.txt').read().split('\n') if parse(r)]
drg = {parse(r)['drg']: parse(r)['desc'] for r in train}
cities = {}
for r in train:
    f = parse(r); cities.setdefault(f['state'], set()).add(f['city'])
cities = {s: sorted(c) for s, c in cities.items()}
rng = random.Random(7)
clean = rng.sample(test, 5)
planted = [corrupt(r, KINDS[i % 4], drg, cities, rng) for i, r in enumerate(rng.sample(test, 5))]
demo = clean + [p for p in planted if p]
rng.shuffle(demo)
open('data/suspicious.txt', 'w').write('\n'.join(demo) + '\n')
open('data/suspicious_answer_key.txt', 'w').write('\n'.join(('PLANTED  ' if r in planted else 'clean    ') + r for r in demo))
!python gpt.py --input data/processed --flag checkpoints/gpt.pth --flag_file data/suspicious.txt   # rules first, then the model
!cat data/suspicious_answer_key.txt

## 7. Results are already in Drive (`build-transformer/medicheck/`)

In [ ]:
!ls -lh checkpoints/ data/described_v2/

## Live demo (Tuesday)
`!python gpt.py --eval checkpoints/explain_v3.pth --interactive --temperature 0.5`  (paste a record, get the plain-English sentence)

`!python gpt.py --input data/processed --flag checkpoints/gpt.pth --flag_file data/suspicious.txt`  (rules, then the model's ranking)